
# Machine Learning aplicado, Clustering 3
## Perfilar inversionistas con una encuesta de hogares: segmentos, patrimonio y los sesgos que esconden

**Curso:** *Machine Learning aplicado* · **Preparado por:** David Díaz, con asistencia de Claude (Anthropic) · **Entorno:** Google Colab

### De dónde venimos

En el primer notebook de clustering segmentamos la base de clientes de un banco: 5.660 filas, ocho columnas, una
tabla limpia. Hoy hacemos lo mismo con una base más seria y con una consecuencia más seria. La base es la **Survey of
Consumer Finances** de la Reserva Federal de Estados Unidos, la mejor encuesta pública sobre las finanzas de los
hogares: patrimonio, ingreso, deudas, qué activos tienen, cuánto saben de finanzas y cuánto riesgo dicen estar
dispuestos a tomar. Y la consecuencia es que una segmentación de inversionistas no se queda en un informe: se usa
para **asignar productos**. Toda institución que asesora inversiones clasifica a sus clientes en un perfil (conservador,
moderado, agresivo); en Chile lo exige la normativa de la CMF, y los multifondos de las AFP hacen lo mismo con
millones de personas.

Dos cosas nuevas respecto del banco. Primero, una encuesta no es una tabla cualquiera: trae **ponderadores** (cada hogar
representa a un número distinto de hogares del país) e **imputaciones múltiples** (cada hogar aparece cinco veces), y si
se ignoran, los números salen mal por un factor grande. Segundo, y es el centro del notebook: los segmentos que
encuentra el algoritmo se parecen mucho a la estructura de riqueza del país, y esa estructura no es neutral respecto
del sexo ni de la raza. Nunca le vamos a dar al algoritmo el sexo ni la raza de nadie, y aun así los segmentos van a
tener composición muy distinta por sexo y raza. Vamos a medir cuánto, y a discutir qué significa.

### Qué vas a aprender hoy

1. Qué es el perfil del inversionista y por qué una segmentación se convierte en una decisión sobre personas.
2. Cómo se trabaja una encuesta con diseño complejo: implicados y ponderadores, y cuánto se equivoca quien los ignora.
3. Cómo preparar variables de dinero con colas larguísimas (logaritmo con signo) y cómo segmentar con ellas.
4. Cómo perfilar segmentos con estadísticas ponderadas, y cómo ponerles nombre.
5. Cómo medir si los segmentos reproducen diferencias por sexo y raza, por qué ocurre aunque esas variables no se usen, y qué se puede hacer con eso.


In [ ]:

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings, os, io, zipfile, urllib.request

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
SEMILLA = 42



## 1. El problema: clasificar personas, no predecir precios

Un asesor de inversiones (una AGF, un banco privado, una AFP) necesita saber qué cartera recomendarle a cada cliente.
La práctica estándar es un cuestionario de perfil y una regla que lo convierte en una categoría. La tentación de
automatizar es obvia: hay datos de miles de clientes, y un algoritmo de clustering puede encontrar "tipos" de
inversionista sin que nadie diseñe el cuestionario.

El problema es que los tipos que encuentra el algoritmo son tipos de **situación financiera**, y la situación
financiera de un país tiene historia. Si los hogares de menor patrimonio quedan en el segmento "conservador" y a ese
segmento se le ofrecen productos de menor retorno esperado, la asignación individual parece razonable y el efecto
agregado es que el acceso al retorno queda condicionado a la riqueza previa. Y como la riqueza previa está
correlacionada con el sexo y la raza por razones que no tienen nada que ver con las preferencias de riesgo, el
algoritmo termina reproduciendo esa brecha sin haber visto nunca una columna de sexo o raza.

Nada de esto es un argumento contra segmentar. Es un argumento a favor de **medir** la composición de los segmentos
antes de usarlos, que es lo que vamos a hacer.

### Los datos

**Survey of Consumer Finances 2022**, extracto resumido (el archivo `SCFP2022` que publica la Reserva Federal): 4.595
hogares, con más de 300 variables. Usaremos unas pocas:


In [ ]:

VARIABLES = {
    "AGE": "edad del jefe de hogar",
    "EDCL": "educación: 1 sin secundaria, 2 secundaria, 3 algo de universidad, 4 título universitario",
    "KIDS": "número de hijos en el hogar",
    "INCOME": "ingreso anual del hogar (USD)",
    "NETWORTH": "patrimonio neto (USD; puede ser negativo)",
    "FIN": "activos financieros (USD)",
    "FINLIT": "alfabetización financiera: respuestas correctas de 0 a 3",
    "NOFINRISK": "1 si el hogar declara no estar dispuesto a tomar ningún riesgo financiero",
    "HEQUITY": "1 si el hogar tiene acciones (directas o vía fondos)",
    "HHSEX": "sexo del jefe de hogar: 1 hombre, 2 mujer",
    "RACECL4": "grupo racial: 1 blanco, 2 negro, 3 hispano, 4 otro",
    "OCCAT1": "ocupación del jefe de hogar: 1 empleado, 2 independiente, 3 jubilado o con discapacidad, 4 no trabaja",
    "MARRIED": "1 casado o conviviente, 2 ni casado ni conviviente",
    "WGT": "ponderador: cuántos hogares del país representa esta fila",
    "YY1": "identificador del hogar",
    "Y1": "identificador del implicado (versión imputada) del hogar",
}
URL_DATOS = ""      # <-- link al archivo del curso (SCFP2022.csv) si lo tienes; si no, se baja de la Reserva Federal (unos 20 MB)
URL_FED = "https://www.federalreserve.gov/econres/files/scfp2022excel.zip"

def normalizar_link(url):
    if "dropbox.com" in url:
        return url.replace("dl=0", "dl=1").replace("www.dropbox.com", "dl.dropboxusercontent.com")
    if "drive.google.com" in url and "/d/" in url:
        return "https://drive.google.com/uc?export=download&id=" + url.split("/d/")[1].split("/")[0]
    return url

if URL_DATOS:
    scf = pd.read_csv(normalizar_link(URL_DATOS))
elif os.path.exists("SCFP2022.csv"):
    scf = pd.read_csv("SCFP2022.csv")
else:
    print("Descargando la SCF 2022 desde la Reserva Federal...")
    peticion = urllib.request.Request(URL_FED, headers={"User-Agent": "Mozilla/5.0"})     # el servidor rechaza peticiones sin User-Agent
    with zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(peticion, timeout=300).read())) as z:
        z.extractall(".")
    scf = pd.read_csv("SCFP2022.csv")
scf = scf[list(VARIABLES)]
print("Filas:", len(scf), "  Hogares distintos:", scf["YY1"].nunique(), "  Filas por hogar:", scf.groupby("YY1").size().unique())
scf.head()



## 2. Dos trampas de la encuesta

### Trampa 1: 22.975 filas que son 4.595 hogares

La SCF publica **cinco versiones de cada hogar**. Muchas respuestas financieras vienen incompletas (la gente no recuerda
el saldo de su cuenta de jubilación, o no quiere decir cuánto tiene), y en vez de descartar esos hogares o rellenar con
el promedio, la Fed genera cinco conjuntos de valores plausibles para lo que falta: cinco **implicados**. La
variabilidad entre ellos mide cuánto no sabemos. Las variables demográficas son iguales en los cinco; las financieras
varían.

Para nosotros tiene una consecuencia inmediata: si segmentamos las 22.975 filas, cada hogar entra cinco veces y los
segmentos quedan inflados por copias. Trabajamos con **el primer implicado de cada hogar** (4.595 filas), y en el
ejercicio 3 comprobamos qué tan distinto saldría con otro.


In [ ]:

ejemplo = scf[scf["YY1"] == scf["YY1"].iloc[0]]
print("Las cinco versiones del primer hogar (las demográficas coinciden, las financieras cambian):")
print(ejemplo[["YY1", "Y1", "AGE", "EDCL", "INCOME", "NETWORTH", "FIN", "WGT"]].to_string(index=False))
hogares = scf.sort_values(["YY1", "Y1"]).groupby("YY1", as_index=False).first()
print()
print("Hogares con los que trabajamos:", len(hogares))



### Trampa 2: los ponderadores

La SCF no es una muestra aleatoria simple: **sobremuestrea a los hogares muy ricos**, porque de otro modo casi no
capturaría la parte alta de la distribución, donde está la mayoría de los activos financieros. Por eso cada fila trae
un ponderador `WGT`, cuántos hogares del país representa. Un promedio sin ponderar describe la muestra (llena de ricos),
no la población. La diferencia no es un detalle:


In [ ]:

def media_ponderada(x, w):
    return np.average(x, weights=w)

def mediana_ponderada(x, w):
    orden = np.argsort(x)
    x_, w_ = np.asarray(x)[orden], np.asarray(w)[orden]
    return np.interp(0.5, np.cumsum(w_) / np.sum(w_), x_)

filas = []
for v in ["INCOME", "NETWORTH", "FIN"]:
    filas.append(dict(variable=v, media_sin_ponderar=hogares[v].mean(), media_ponderada=media_ponderada(hogares[v], hogares["WGT"]),
                      mediana_sin_ponderar=hogares[v].median(), mediana_ponderada=mediana_ponderada(hogares[v], hogares["WGT"])))
tabla = pd.DataFrame(filas).set_index("variable")
tabla["la media sin ponderar es ... veces la real"] = tabla["media_sin_ponderar"] / tabla["media_ponderada"]
print(tabla.round(0).to_string())



El patrimonio medio sin ponderar es más de diez veces el real, y la mediana sin ponderar también está muy por encima:
la muestra está llena de hogares ricos a propósito. La mediana ponderada de patrimonio, cerca de 190.000 dólares, es
la cifra que la Fed reporta para 2022. **Regla:** antes de tocar una encuesta, busca el ponderador y cuenta los
implicados.

Para el clustering hay un matiz. K-means acepta ponderadores (`sample_weight`) y los vamos a usar para que los
centroides representen a la población y no a la muestra de ricos. Pero las **estadísticas descriptivas** de cada
segmento (edad promedio, patrimonio mediano, porcentaje de mujeres) hay que calcularlas siempre ponderadas, y eso lo
hacemos a mano con las dos funciones de arriba.

## 3. Preparar las variables

Las tres variables de dinero tienen colas enormes (el patrimonio va de −555.000 a más de 2.000 millones de dólares).
Con esas colas, k-means dedicaría todos sus grupos a separar a los pocos hogares de la punta. La transformación
estándar es el **logaritmo con signo**, $\operatorname{sign}(x)\,\ln(1 + |x|)$, que comprime las colas y acepta ceros
y negativos. Después escalamos, como siempre.

La elección de variables es una decisión que hay que declarar: usamos edad, educación, hijos, alfabetización
financiera y las tres de dinero. **No usamos el sexo ni la raza**, y no las usamos a propósito, para poder preguntar
después si los segmentos las reconstruyen solos.

La misma regla del notebook anterior aplica a cómo entran las variables. `EDCL` es una escala de 1 a 4 que la propia
encuesta define, y la usamos como tal con el supuesto de saltos iguales declarado; `KIDS` y `FINLIT` son conteos. Las
variables nominales de la encuesta (ocupación `OCCAT1`, estado civil `MARRIED`, tenencia de vivienda `HOUSECL`) quedan
fuera de la segmentación, y si quisiéramos incluirlas haríamos lo que hicimos con el banco: definir un eje de negocio
y asignar puntajes a mano, nunca one-hot a ciegas, que pondría a un hogar casado tan lejos de uno separado como de uno
soltero y multiplicaría las dimensiones. El ejercicio 4 lo hace.


In [ ]:

datos = hogares.copy()
for v in ["INCOME", "NETWORTH", "FIN"]:
    datos["log_" + v] = np.sign(datos[v]) * np.log1p(np.abs(datos[v]))
SEGMENTAR = ["AGE", "EDCL", "KIDS", "FINLIT", "log_INCOME", "log_NETWORTH", "log_FIN"]
Z = StandardScaler().fit_transform(datos[SEGMENTAR])
w = datos["WGT"].values
fig = px.histogram(datos, x="log_NETWORTH", nbins=60, title="Patrimonio neto en logaritmo con signo (negativo = deudas mayores que activos)")
fig.update_layout(height=340, width=720)
fig.show()



## 4. Cuántos segmentos

Los mismos dos criterios del notebook anterior, con el ponderador en el ajuste.


In [ ]:

filas = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(Z, sample_weight=w)
    filas.append(dict(K=k, inercia=km.inertia_, silueta=silhouette_score(Z, km.labels_, sample_size=3000, random_state=SEMILLA)))
criterios = pd.DataFrame(filas).set_index("K")
print(criterios.round(3).to_string())
fig = go.Figure()
fig.add_scatter(x=criterios.index, y=criterios["inercia"], mode="lines+markers", name="inercia")
fig.add_scatter(x=criterios.index, y=criterios["silueta"], mode="lines+markers", name="silueta", yaxis="y2")
fig.update_layout(title="Inercia y silueta según el número de segmentos", xaxis_title="K", yaxis=dict(title="inercia"), yaxis2=dict(title="silueta", overlaying="y", side="right"), height=400, width=720)
fig.show()



Ni la inercia muestra un codo ni la silueta un máximo claro: los hogares forman un **continuo** de situaciones
financieras, no islas, y cualquier $K$ es un corte de ese continuo. Es lo normal con datos de personas. Elegimos
**K = 4** porque produce segmentos que se pueden describir y porque es el número de perfiles que suele usar la industria
(conservador, moderado, agresivo, más uno de "sin capacidad de ahorro"). Con K = 6 el ejercicio 1 revisa si las
conclusiones cambian.


In [ ]:

K = 4
modelo = KMeans(n_clusters=K, n_init=20, random_state=SEMILLA).fit(Z, sample_weight=w)
datos["segmento"] = modelo.labels_

def perfil_segmentos(d, col="segmento"):
    g = d.groupby(col)
    return pd.DataFrame({
        "hogares en la muestra": g.size(),
        "% de la población": g["WGT"].sum() / d["WGT"].sum(),
        "edad": g.apply(lambda x: media_ponderada(x["AGE"], x["WGT"])),
        "educación (1 a 4)": g.apply(lambda x: media_ponderada(x["EDCL"], x["WGT"])),
        "hijos": g.apply(lambda x: media_ponderada(x["KIDS"], x["WGT"])),
        "alfab. financiera (0 a 3)": g.apply(lambda x: media_ponderada(x["FINLIT"], x["WGT"])),
        "ingreso mediano": g.apply(lambda x: mediana_ponderada(x["INCOME"], x["WGT"])),
        "patrimonio mediano": g.apply(lambda x: mediana_ponderada(x["NETWORTH"], x["WGT"])),
        "activos fin. medianos": g.apply(lambda x: mediana_ponderada(x["FIN"], x["WGT"])),
        "% sin tolerancia al riesgo": g.apply(lambda x: media_ponderada(x["NOFINRISK"], x["WGT"])),
        "% con acciones": g.apply(lambda x: media_ponderada(x["HEQUITY"], x["WGT"])),
    })

perfil = perfil_segmentos(datos)
print(perfil.round(2).to_string())



## 5. Leer los segmentos

Cuatro perfiles claros, ordenados por patrimonio y etapa de la vida (los números de segmento son arbitrarios; los
nombres los ponemos nosotros mirando la tabla):

- **Patrimonio consolidado** (32% de la población): 62 años en promedio, patrimonio mediano de 850.000 dólares y
  380.000 en activos financieros, la educación y la alfabetización financiera más altas, 87% con acciones y solo uno
  de cada cinco se declara sin tolerancia al riesgo.
- **Profesionales con familia** (34%): 36 años, 1,4 hijos, ingreso mediano de 80.000 dólares, patrimonio en
  construcción (122.000), educación alta, 62% con acciones, 28% sin tolerancia al riesgo.
- **Patrimonio bajo** (27%): 65 años, ingreso mediano de 38.000 dólares, apenas 4.000 en activos financieros (el
  patrimonio mediano de 97.000 es casi todo la vivienda), la educación y la alfabetización más bajas, 26% con acciones
  y **68% sin tolerancia al riesgo**.
- **Endeudados jóvenes** (8%): 41 años, patrimonio neto **negativo**, casi sin activos financieros, 48% sin
  tolerancia al riesgo.

La aversión al riesgo declarada sigue al patrimonio: 21 a 28% en los dos segmentos con dinero, 48 a 68% en los dos
sin dinero. Es el resultado que haría que un asesor automatizado ofreciera productos conservadores a quienes menos
tienen, y es el punto de partida de la sección siguiente.


In [ ]:

NOMBRES = perfil["patrimonio mediano"].rank().map({4.0: "patrimonio consolidado", 3.0: "profesionales con familia", 2.0: "patrimonio bajo", 1.0: "endeudados jóvenes"})
datos["perfil"] = datos["segmento"].map(NOMBRES)
ORDEN = ["patrimonio consolidado", "profesionales con familia", "patrimonio bajo", "endeudados jóvenes"]

centroides = pd.DataFrame(modelo.cluster_centers_, columns=SEGMENTAR)
centroides.insert(0, "perfil", [NOMBRES[k] for k in range(K)])
fig = px.parallel_coordinates(centroides.assign(id=range(K)), color="id", dimensions=SEGMENTAR, title="Centroides en unidades estandarizadas (0 = promedio; cada línea es un segmento)")
fig.update_layout(height=440)
fig.show()
muestra = datos.sample(2500, random_state=SEMILLA)
fig = px.scatter(muestra, x="AGE", y="log_NETWORTH", color="perfil", category_orders={"perfil": ORDEN}, opacity=0.55,
                 labels={"AGE": "edad", "log_NETWORTH": "patrimonio neto (log con signo)"}, title="Los segmentos en el plano edad y patrimonio (2.500 hogares al azar)")
fig.update_layout(height=460, width=820)
fig.show()


In [ ]:

resumen = perfil.copy(); resumen.index = [NOMBRES[k] for k in resumen.index]; resumen = resumen.loc[ORDEN]
fig = go.Figure()
fig.add_bar(x=ORDEN, y=resumen["% sin tolerancia al riesgo"], name="% sin tolerancia al riesgo")
fig.add_bar(x=ORDEN, y=resumen["% con acciones"], name="% con acciones")
fig.add_hline(y=media_ponderada(datos["NOFINRISK"], datos["WGT"]), line=dict(dash="dot"), annotation_text="promedio nacional sin tolerancia")
fig.update_layout(barmode="group", title="Actitud y tenencia de riesgo por segmento (ponderado)", height=400, width=760, yaxis_tickformat=".0%")
fig.show()



## 6. Los sesgos que esconden los segmentos

Ahora la pregunta central. Nunca usamos el sexo ni la raza. ¿Los segmentos son neutrales respecto de ellos?

Dos formas de mirarlo, y conviene hacer las dos. La **composición** de cada segmento (qué porcentaje de sus hogares
tiene jefa de hogar mujer, o es negro o hispano) comparada con la población; y la **distribución** de cada grupo entre
los segmentos (de todos los hogares hispanos, qué fracción cae en "patrimonio bajo"). La segunda es la que importa
para una regla de asignación: si al segmento "patrimonio consolidado" se le ofrecen los productos de mayor retorno
esperado, esa fracción es la **tasa de acceso** de cada grupo a esos productos.


In [ ]:

datos["sexo"] = datos["HHSEX"].map({1: "hombre", 2: "mujer"})
datos["raza"] = datos["RACECL4"].map({1: "blanco", 2: "negro", 3: "hispano", 4: "otro"})

def composicion(d, grupo):
    t = d.groupby(["perfil", grupo])["WGT"].sum().unstack(grupo)
    t = t.div(t.sum(axis=1), axis=0).loc[ORDEN]
    t.loc["población"] = d.groupby(grupo)["WGT"].sum() / d["WGT"].sum()
    return t

def distribucion(d, grupo):
    t = d.groupby([grupo, "perfil"])["WGT"].sum().unstack("perfil")[ORDEN]
    return t.div(t.sum(axis=1), axis=0)

print("Composición de cada segmento por sexo del jefe de hogar (ponderada), contra la población:")
print(composicion(datos, "sexo").round(2).to_string())
print()
print("Composición por grupo racial:")
print(composicion(datos, "raza").round(2).to_string())
print()
print("Distribución de cada grupo entre los segmentos (cada fila suma 1):")
dist_sexo, dist_raza = distribucion(datos, "sexo"), distribucion(datos, "raza")
print(pd.concat([dist_sexo, dist_raza]).round(2).to_string())


In [ ]:

largo = pd.concat([dist_sexo, dist_raza]).reset_index().melt(id_vars="index", var_name="perfil", value_name="fraccion").rename(columns={"index": "grupo"})
fig = px.bar(largo, x="perfil", y="fraccion", color="grupo", barmode="group", category_orders={"perfil": ORDEN},
             title="Qué fracción de cada grupo cae en cada segmento (ponderado)")
fig.update_layout(height=430, width=860, yaxis_tickformat=".0%")
fig.show()



Los números son grandes y conviene decirlos sin rodeos. El 35% de los hogares con jefa de hogar mujer cae en
"patrimonio bajo" y el 15% en "endeudados jóvenes", contra 23% y 5% de los hogares con jefe hombre; en "patrimonio
consolidado" la relación se invierte (24% contra 35%). Entre los hogares hispanos, el 36% está en "patrimonio bajo" y
solo el 11% en "patrimonio consolidado"; entre los hogares blancos, 26% y 38%. Y el segmento de los endeudados tiene
mayoría de jefas de hogar (53%) y una de cada cuatro familias es negra, el doble de su peso en la población. Si la
regla fuera "productos de mayor retorno esperado para el segmento consolidado", la tasa de acceso de un hogar blanco
sería **tres veces y media** la de un hogar hispano.

Esto no lo hizo el algoritmo: lo hizo la **estructura de riqueza** del país, que el algoritmo describió con precisión.
Las variables que usamos (patrimonio, ingreso, educación, alfabetización financiera) son **proxies** del sexo y de la
raza porque en Estados Unidos, y en Chile, la riqueza está correlacionada con ellos por razones históricas. Borrar las
columnas protegidas no borra su efecto; solo impide medirlo. Lo medimos con una prueba y un tamaño de efecto:


In [ ]:

def cramer_v(tabla):
    chi2 = stats.chi2_contingency(tabla)[0]
    n = tabla.values.sum()
    return np.sqrt(chi2 / (n * (min(tabla.shape) - 1)))

for grupo in ["sexo", "raza"]:
    tabla = pd.crosstab(datos["perfil"], datos[grupo])              # conteos de la muestra (la prueba necesita conteos, no pesos)
    chi2, p, gl, _ = stats.chi2_contingency(tabla)
    print(f"Segmento x {grupo}: chi-cuadrado {chi2:.0f}, valor p {p:.1e}, V de Cramér {cramer_v(tabla):.3f}")
print()
print("La V de Cramér va de 0 (independencia) a 1 (determinación total). Valores de 0,1 a 0,3 son asociaciones")
print("moderadas: los segmentos no determinan el sexo ni la raza, pero están lejos de ser independientes de ellos.")



### Qué se hace con esto

Hay tres respuestas, y las tres son decisiones de política, no de modelación:

1. **Medir y reportar.** Lo mínimo exigible: toda segmentación que se use para asignar productos debería venir con su
   tabla de composición por grupos protegidos. Para eso hay que **tener** las variables protegidas en la base, aunque
   no se usen para segmentar. La intuición de "mejor no recolectarlas" produce segmentaciones igual de sesgadas e
   imposibles de auditar.
2. **Separar tolerancia de capacidad.** El 59% "sin tolerancia al riesgo" del segmento de patrimonio bajo mezcla dos
   cosas: hogares que no quieren riesgo y hogares que no pueden absorber pérdidas. Un cuestionario de perfil debe
   preguntar las dos por separado, y la recomendación debería depender de la segunda tanto como de la primera.
3. **Definir qué significa "justo" antes de usar la regla.** Si la asignación exige igualdad de tasas de acceso entre
   grupos (paridad demográfica), habrá que intervenir sobre la regla; si exige que a igual situación financiera se
   recomiende lo mismo, la regla actual ya lo cumple y la brecha es la del país. Las dos definiciones son razonables y
   son incompatibles; elegir una es una decisión que el algoritmo no puede tomar.

## 7. Lo que hay que llevarse

1. Una encuesta trae ponderadores e implicados. Ignorarlos infla el patrimonio medio más de diez veces y cuenta cada hogar cinco veces.
2. Las variables de dinero se segmentan en logaritmo con signo; sin eso k-means solo separa a los más ricos.
3. Con datos de hogares no hay un K correcto: hay un continuo y un corte útil. K = 4 da perfiles que la industria reconoce.
4. Los segmentos siguen al patrimonio, y la aversión al riesgo declarada también: 21 a 28% en los segmentos con dinero, 48 a 68% en los sin dinero.
5. Sin usar sexo ni raza, los segmentos tienen composición muy distinta por sexo y raza, porque las variables financieras son proxies. La tasa de acceso al segmento "consolidado" es tres veces y media mayor para un hogar blanco que para uno hispano.
6. Borrar las variables protegidas no corrige nada; medir y reportar sí. Qué hacer después es una decisión de política.



## Ejercicios

1. **Más segmentos.** Repite la segmentación con K = 6, nómbralos, y recalcula la distribución de cada grupo racial
   entre segmentos. ¿La brecha de acceso al segmento de mayor patrimonio se achica, se agranda o se reparte?
2. **Sin dinero.** Segmenta usando solo edad, educación, hijos y alfabetización financiera (sin ingreso, patrimonio ni
   activos). ¿Los segmentos siguen teniendo composición distinta por sexo y raza? ¿Qué variable queda haciendo de
   proxy?
3. **Otro implicado.** Repite la segmentación con el segundo implicado de cada hogar (`Y1` terminado en 2) y compara
   con la del primero usando el índice de Rand ajustado. ¿Cuánto cambia la asignación de un hogar según qué versión
   imputada le tocó?
4. **Agregar categóricas con puntajes.** Incluye la ocupación (`OCCAT1`) y el estado civil (`MARRIED`) en la
   segmentación, como hicimos con el banco: define ejes de negocio (ingreso esperado y estabilidad para la ocupación,
   compromiso del hogar para el estado civil), declara los puntajes, vuelve a segmentar con K = 4 y compara con la
   segmentación base (Rand ajustado, perfiles) y, sobre todo, recalcula la V de Cramér con el sexo y la raza. ¿Agregar
   variables que no son el sexo cambia cuánto se asocian los segmentos con el sexo?



## Soluciones


In [ ]:

# Ejercicio 1: K = 6
m6 = KMeans(n_clusters=6, n_init=20, random_state=SEMILLA).fit(Z, sample_weight=w)
d6 = datos.copy(); d6["segmento"] = m6.labels_
p6 = perfil_segmentos(d6)
p6 = p6.sort_values("patrimonio mediano", ascending=False)
print(p6[["% de la población", "edad", "ingreso mediano", "patrimonio mediano", "% sin tolerancia al riesgo", "% con acciones"]].round(2).to_string())
d6["perfil"] = d6["segmento"].map({s: f"seg {s} (patrimonio mediano {p6.loc[s, 'patrimonio mediano']:,.0f})" for s in p6.index})
orden6 = [f"seg {s} (patrimonio mediano {p6.loc[s, 'patrimonio mediano']:,.0f})" for s in p6.index]
t = d6.groupby(["raza", "perfil"])["WGT"].sum().unstack("perfil")[orden6]
print()
print("Distribución de cada grupo racial entre los seis segmentos (ordenados de más a menos patrimonio):")
print(t.div(t.sum(axis=1), axis=0).round(2).to_string())



Con seis segmentos la brecha no desaparece: se reparte. El segmento de mayor patrimonio sigue concentrando hogares
blancos muy por encima de su peso poblacional, y los dos segmentos de menor patrimonio siguen concentrando a los
hogares negros e hispanos. Cambiar $K$ cambia la granularidad de la descripción, no la estructura que describe.


In [ ]:

# Ejercicio 2: segmentar sin variables de dinero
SIN_DINERO = ["AGE", "EDCL", "KIDS", "FINLIT"]
Z2 = StandardScaler().fit_transform(datos[SIN_DINERO])
m2 = KMeans(n_clusters=K, n_init=20, random_state=SEMILLA).fit(Z2, sample_weight=w)
d2 = datos.copy(); d2["segmento"] = m2.labels_
p2 = perfil_segmentos(d2)
d2["perfil"] = d2["segmento"].map({s: f"seg {s}" for s in p2.index})
print(p2[["% de la población", "edad", "educación (1 a 4)", "hijos", "alfab. financiera (0 a 3)", "patrimonio mediano", "% sin tolerancia al riesgo"]].round(2).to_string())
for grupo in ["sexo", "raza"]:
    tabla = pd.crosstab(d2["perfil"], d2[grupo])
    print(f"V de Cramér segmento x {grupo}: {cramer_v(tabla):.3f}   (con dinero era {cramer_v(pd.crosstab(datos['perfil'], datos[grupo])):.3f})")
t = d2.groupby(["raza", "perfil"])["WGT"].sum().unstack("perfil")
print()
print(t.div(t.sum(axis=1), axis=0).round(2).to_string())



Sin las variables de dinero la asociación con la raza baja pero no desaparece, porque la educación y la alfabetización
financiera también están correlacionadas con el grupo racial. Y los segmentos resultantes son peores para el
propósito original (ya no separan por patrimonio, que es lo que importa para asignar productos). Sacar variables no es
una solución: cambia qué proxy hace el trabajo.


In [ ]:

# Ejercicio 3: el segundo implicado
segundo = scf.sort_values(["YY1", "Y1"]).groupby("YY1", as_index=False).nth(1).set_index("YY1").loc[hogares["YY1"]].reset_index()
d3 = segundo.copy()
for v in ["INCOME", "NETWORTH", "FIN"]:
    d3["log_" + v] = np.sign(d3[v]) * np.log1p(np.abs(d3[v]))
Z3 = StandardScaler().fit_transform(d3[SEGMENTAR])
etiquetas_3 = modelo.predict(Z3)                      # mismos centroides, otra versión imputada de cada hogar
print("Hogares que cambian de segmento entre el primer y el segundo implicado:", f"{(etiquetas_3 != modelo.labels_).mean():.1%}")
m3 = KMeans(n_clusters=K, n_init=20, random_state=SEMILLA).fit(Z3, sample_weight=d3["WGT"].values)
print("Rand ajustado entre una segmentación hecha con el implicado 1 y otra hecha con el implicado 2:", round(adjusted_rand_score(modelo.labels_, m3.labels_), 3))



La gran mayoría de los hogares queda en el mismo segmento con cualquiera de sus versiones imputadas, y las dos
segmentaciones casi coinciden: la imputación mueve a los hogares cercanos a una frontera, no la estructura. Para un
informe serio lo correcto es repetir el análisis con los cinco implicados y reportar el rango; para la clase, el
primero basta y la comprobación de este ejercicio es la que da la tranquilidad.


In [ ]:

# Ejercicio 4: ocupación y estado civil como puntajes sobre ejes declarados (misma lógica que en el notebook del banco)
PUNTAJE_INGRESO_OCUP = {1: 60, 2: 75, 3: 30, 4: 0}       # empleado, independiente, jubilado, no trabaja
PUNTAJE_ESTAB_OCUP = {1: 80, 2: 40, 3: 100, 4: 0}        # una pensión es el ingreso más previsible; no trabajar, el menos
PUNTAJE_COMPROMISO = {1: 100, 2: 0}                       # casado o conviviente, contra ni uno ni otro
d4 = datos.copy()
d4["ingreso_ocup"] = d4["OCCAT1"].map(PUNTAJE_INGRESO_OCUP)
d4["estab_ocup"] = d4["OCCAT1"].map(PUNTAJE_ESTAB_OCUP)
d4["compromiso"] = d4["MARRIED"].map(PUNTAJE_COMPROMISO)
SEGMENTAR_4 = SEGMENTAR + ["ingreso_ocup", "estab_ocup", "compromiso"]
Z4 = StandardScaler().fit_transform(d4[SEGMENTAR_4])
m4 = KMeans(n_clusters=K, n_init=20, random_state=SEMILLA).fit(Z4, sample_weight=w)
d4["perfil4"] = m4.labels_
print("Rand ajustado contra la segmentación base:", round(adjusted_rand_score(modelo.labels_, m4.labels_), 3))
g = d4.groupby("perfil4")
print(pd.DataFrame({"% de la población": g["WGT"].sum() / d4["WGT"].sum(), "edad": g.apply(lambda x: media_ponderada(x["AGE"], x["WGT"])),
                    "patrimonio mediano": g["NETWORTH"].median(), "ingreso ocup.": g.apply(lambda x: media_ponderada(x["ingreso_ocup"], x["WGT"])),
                    "estabilidad ocup.": g.apply(lambda x: media_ponderada(x["estab_ocup"], x["WGT"])), "compromiso": g.apply(lambda x: media_ponderada(x["compromiso"], x["WGT"])),
                    "% sin tolerancia al riesgo": g.apply(lambda x: media_ponderada(x["NOFINRISK"], x["WGT"]))}).round(2).to_string())
print()
for grupo in ["sexo", "raza"]:
    print(f"V de Cramér segmento x {grupo}: base {cramer_v(pd.crosstab(datos['perfil'], datos[grupo])):.3f}   con ocupación y estado civil {cramer_v(pd.crosstab(d4['perfil4'], d4[grupo])):.3f}")
print()
print("Composición por sexo de los nuevos segmentos (ponderada):")
t = d4.groupby(["perfil4", "sexo"])["WGT"].sum().unstack("sexo")
print(t.div(t.sum(axis=1), axis=0).round(2).to_string())



Con los puntajes de ocupación y estado civil la segmentación cambia de fondo (Rand ajustado de 0,25): aparece un
segmento de jubilados (28% de los hogares, 72 años, estabilidad 99), uno chico de hogares donde el jefe no trabaja
(4%), y los otros dos se ordenan por si el hogar está en pareja o no, con el patrimonio siguiéndolos. Y el resultado
que importa para el tema de este notebook: la asociación con la raza baja un poco (V de Cramér de 0,25 a 0,22), pero
la asociación con el **sexo** sube (de 0,31 a 0,38), y el segmento de mayor patrimonio queda con 90% de jefes de hogar
hombres, porque en la encuesta el "jefe" de un hogar en pareja es casi siempre el hombre y el de un hogar sin pareja
es con mucha frecuencia una mujer. El eje de compromiso, definido con toda la lógica de negocio del mundo, es un proxy
del sexo del jefe de hogar. Es la misma lección de la sección 5 con otra variable: declarar los supuestos del mapeo es
necesario, pero no garantiza neutralidad; hay que medirla después, cada vez.
